In [2]:
import torch
import torch.nn as nn
import json
import re
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

with open("../../backend/data/processed/new_train.json", "r", encoding='utf-8') as f:
    train = json.load(f)
with open('../../backend/data/processed/new_validation.json', 'r', encoding = 'utf-8') as f:
    validation = json.load(f)
with open('../../backend/data/processed/new_test.json', 'r', encoding='utf-8') as f:
    test = json.load(f)

print(len(train))
print(len(test))
print(len(validation))

print(train[100])
print(type(train))
print(type(train[0]))

1291
282
274
{'text': 'Bị ấm đầu, hắt hơi liên tục', 'label': 'health'}
<class 'list'>
<class 'dict'>


In [3]:
def tokenizer(text):
    text = text.lower()
    text = re.sub(r'[.!?,;:]', '', text)
    return text.split()

In [4]:
from collections import Counter
counter = Counter()
for item in train:
    tokens = tokenizer((item['text']))
    counter.update(tokens)
print(counter.most_common(100))
print(len(counter))

[('ăn', 359), ('bé', 259), ('con', 208), ('bị', 161), ('chơi', 161), ('không', 154), ('ngủ', 154), ('trẻ', 153), ('hết', 143), ('tự', 104), ('bạn', 103), ('xúc', 97), ('cô', 95), ('cơm', 94), ('nay', 88), ('một', 86), ('mình', 85), ('bát', 81), ('với', 79), ('ngoan', 74), ('mới', 73), ('đầu', 72), ('khi', 71), ('em', 67), ('chịu', 61), ('trưa', 59), ('giờ', 58), ('bánh', 57), ('động', 56), ('đau', 55), ('cực', 55), ('đồ', 52), ('hào', 51), ('tích', 51), ('khóc', 50), ('tập', 49), ('trung', 49), ('ngồi', 48), ('phải', 46), ('nhảy', 46), ('bỏ', 46), ('hứng', 45), ('cùng', 45), ('rồi', 45), ('ho', 44), ('ra', 44), ('sốt', 43), ('đi', 43), ('xếp', 42), ('mứa', 42), ('được', 39), ('hôm', 39), ('tay', 39), ('ngon', 38), ('đòi', 38), ('gần', 36), ('ít', 36), ('năng', 34), ('lắm', 34), ('nuốt', 33), ('hình', 33), ('lơ', 33), ('cho', 33), ('sổ', 32), ('mũi', 32), ('mệt', 32), ('trong', 31), ('lên', 31), ('đễnh', 31), ('sữa', 30), ('nhè', 30), ('vui', 30), ('thịt', 30), ('ấm', 30), ('nằm', 30), 

In [5]:
max_vocab_size = 20000
vocab = {
    '<pad>':0,
    '<unk>':1
}
for word , counter in counter.most_common(max_vocab_size):
    vocab[word] = len(vocab)
print(len(vocab))

956


In [6]:
print("Number of samples:", len(train))
print("Vocab size:", len(vocab))
print(train[0])
print(train[1])
print(train[2])

Number of samples: 1291
Vocab size: 956
{'text': ' Tính nay quậy phá', 'label': 'mood'}
{'text': 'Con bị sốt cao, lừ đừ', 'label': 'health'}
{'text': ' Ngậm cơm hoài, chẳng nuốt được bao nhiêu', 'label': 'meals'}


In [7]:
all_tokens = []

for item in train:
    tokens = item["text"].split()
    all_tokens.extend(tokens)

freq = Counter(all_tokens)

print("Number of tokens:", len(all_tokens))
print("Vocab:", len(freq))
print("Freq = 1:", sum(1 for x in freq.values() if x == 1))
print("Freq <= 2:", sum(1 for x in freq.values() if x <= 2))

Number of tokens: 9506
Vocab: 1126
Freq = 1: 364
Freq <= 2: 549


In [8]:
def numericalize(text):
    tokens = tokenizer(text)
    return[
        vocab.get(token, vocab[('<unk>')])
        for token in tokens
    ]
text = "trưa nay bé Kiệt ăn không hết phần cơm, ngủ ít"
numericalized = numericalize(text)
print(numericalized)

[27, 16, 3, 1, 2, 7, 10, 206, 15, 8, 58]


In [9]:
label_to_id = {
    'meals' : 0,
    'sleeping' : 1,
    'health' : 2,
    'activities' : 3,
    'mood' : 4
}

In [10]:
from collections import Counter

num_classes = len(label_to_id)
total_sample = len(train)
class_weight = []


In [11]:
class kindergatern_dataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, index):
        item = self.dataset[index]
        text = numericalize(item['text'])
        label = label_to_id[item['label']]
        return torch.tensor(text), torch.tensor(label)
        

In [12]:
def collate_fn(batch):
    texts, labels = zip(*batch)
    texts = pad_sequence(texts, batch_first=True, padding_value=vocab["<pad>"])
    labels = torch.stack(labels)
    return texts, labels
train_dataset = kindergatern_dataset(train)
validation_dataset = kindergatern_dataset(validation)
test_dataset = kindergatern_dataset(test)
print(type(train_dataset))
print(type(train_dataset.dataset))
print(type(train_dataset.dataset[0]))
print(train_dataset.dataset[0])


<class '__main__.kindergatern_dataset'>
<class 'list'>
<class 'dict'>
{'text': ' Tính nay quậy phá', 'label': 'mood'}


In [13]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle = True,
    collate_fn = collate_fn
)

validation_dataloader = DataLoader(
    validation_dataset,
    batch_size = 32,
    shuffle = False,
    collate_fn = collate_fn
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size = 32,
    shuffle = False,
    collate_fn = collate_fn
)

In [14]:
texts, labels = next(iter(train_dataloader))
print(texts.shape)
print(labels.shape)
embedding_dim = 128

torch.nn.Embedding(
    max_vocab_size, embedding_dim
)

torch.Size([32, 13])
torch.Size([32])


Embedding(20000, 128)

In [15]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super(). __init__()
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx = 0
        )
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first = True
        )
        self.fc = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        embedd = self.embedding(x)
        out_put, (hidden, cell) = self.lstm(embedd)
        hidden = hidden[-1]
        out_put = self.fc(hidden)
        return out_put

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = LSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim = embedding_dim,
    hidden_dim = 64,
    num_classes = 5
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

cuda


In [17]:
num_epoch = 5
for epoch in range(num_epoch):
    model.train()
    total_loss = 0
    for texts, labels in train_dataloader:
        texts = texts.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        out_put = model(texts)
        loss = criterion(out_put, labels)
        loss.backward()
        optimizer.step()
        total_loss = total_loss + loss.item()
    print(f"Epoch {epoch + 1}/{num_epoch}, Loss: {loss.item():.4f}")


Epoch 1/5, Loss: 1.2938
Epoch 2/5, Loss: 0.9806
Epoch 3/5, Loss: 1.1249
Epoch 4/5, Loss: 0.3117
Epoch 5/5, Loss: 0.0840


In [18]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for texts, labels in test_dataloader:

        texts = texts.to(device)
        labels = labels.to(device)

        outputs = model(texts)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total

print(f"Validation Accuracy: {accuracy:.4f}")

Validation Accuracy: 0.8440


In [20]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for texts, labels in test_dataloader:
        texts = texts.to(device)
        labels = labels.to(device)

        outputs = model(texts)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_accuracy = correct / total

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Total Test Samples: {total}")

Test Accuracy: 0.8440
Test Accuracy: 84.40%
Total Test Samples: 282
